# Evaluate DeNICE task 5, round 19 on Kaggle

Attach two Kaggle Input datasets before running: (1) the `100-clients` data directory, and (2) one ZIP archive containing `checkpoint_task_5_base.pt` plus `checkpoint_task_5_round_0.pt` through `checkpoint_task_5_round_19.pt`. Enable Internet so the notebook can clone the matching evaluator code.


In [ ]:
# Cell 1 — clone evaluator code
from pathlib import Path
import shutil
import subprocess
import sys

REPO_PATH = Path('/kaggle/working/FL_IL_IDS')
if REPO_PATH.exists():
    shutil.rmtree(REPO_PATH)
subprocess.run([
    'git', 'clone', '--depth', '1',
    'https://github.com/khoilv2005/FL_IL_IDS.git', str(REPO_PATH),
], check=True)
print('Repository:', subprocess.check_output(['git', '-C', str(REPO_PATH), 'rev-parse', '--short', 'HEAD'], text=True).strip())


In [ ]:
# Cell 2 — configure Kaggle Input paths
# Change DATA_DIR if the dataset slug/folder name differs in your Kaggle Input.
KAGGLE_INPUT = Path('/kaggle/input')
DATA_DIR = Path('/kaggle/input/100-clients/100-clients')  # <-- sửa nếu cần
CHECKPOINT_NAME = 'checkpoint_task_5_round_19.pt'
CHECKPOINT_ARCHIVE_PATH = None  # Set exact .zip path here if Kaggle Input contains multiple ZIP files.

if not DATA_DIR.is_dir():
    raise FileNotFoundError(f'DATA_DIR does not exist: {DATA_DIR}. Edit Cell 2.')

if CHECKPOINT_ARCHIVE_PATH is None:
    archive_candidates = sorted(KAGGLE_INPUT.rglob('*.zip'))
    if len(archive_candidates) != 1:
        print('ZIP files found:', *archive_candidates, sep='\n  ')
        raise RuntimeError('Attach exactly one checkpoint ZIP, or set CHECKPOINT_ARCHIVE_PATH explicitly.')
    CHECKPOINT_ARCHIVE_PATH = archive_candidates[0]
else:
    CHECKPOINT_ARCHIVE_PATH = Path(CHECKPOINT_ARCHIVE_PATH)

if not CHECKPOINT_ARCHIVE_PATH.is_file():
    raise FileNotFoundError(f'Checkpoint ZIP does not exist: {CHECKPOINT_ARCHIVE_PATH}')
print('Dataset:', DATA_DIR)
print('Checkpoint ZIP:', CHECKPOINT_ARCHIVE_PATH)


In [ ]:
# Cell 3 — safely extract the delta checkpoint chain to writable Kaggle storage
import zipfile

EXTRACT_DIR = Path('/kaggle/working/denice_task5_checkpoint')
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True)

with zipfile.ZipFile(CHECKPOINT_ARCHIVE_PATH, 'r') as archive:
    extract_root = EXTRACT_DIR.resolve()
    for member in archive.infolist():
        target = (EXTRACT_DIR / member.filename).resolve()
        if target != extract_root and extract_root not in target.parents:
            raise RuntimeError(f'Unsafe ZIP member: {member.filename}')
    archive.extractall(EXTRACT_DIR)

checkpoint_candidates = sorted(EXTRACT_DIR.rglob('checkpoint_task_5_round_19.pt'))
if len(checkpoint_candidates) != 1:
    raise RuntimeError(f'Expected exactly one final checkpoint; found: {checkpoint_candidates}')
CHECKPOINT_PATH = checkpoint_candidates[0]
print('Checkpoint extracted to:', CHECKPOINT_PATH)


In [ ]:
# Cell 4 — validate task 5 / round 19 and every required delta dependency
import torch

header = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
assert header.get('algorithm') == 'denice', header.get('algorithm')
assert int(header.get('task_id', header.get('task', -1))) == 5, header.get('task_id')
assert int(header.get('round_id', header.get('final_round_id', -1))) == 19, header.get('round_id')

if header.get('checkpoint_type') == 'denice_delta_round':
    required = [CHECKPOINT_PATH.parent / header['base_path']]
    cursor = header
    while cursor.get('previous_round_path'):
        previous = CHECKPOINT_PATH.parent / cursor['previous_round_path']
        required.append(previous)
        if not previous.is_file():
            raise FileNotFoundError(f'Missing required delta: {previous}')
        cursor = torch.load(previous, map_location='cpu', weights_only=False)
    print(f'Delta chain verified: {len(required)} dependency files.')
else:
    print('Full task-end checkpoint verified.')


In [ ]:
# Cell 5 — official cumulative evaluation over all classes from task 0 through task 5
EVAL_OUTPUT = Path('/kaggle/working/denice_task5_round19_eval.json')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
command = [
    sys.executable, str(REPO_PATH / 'eval_checkpoint.py'),
    '--checkpoint', str(CHECKPOINT_PATH),
    '--data-dir', str(DATA_DIR),
    '--device', device,
    '--router-mode', 'multiclass',
    '--evaluation-mode', 'representative',
    '--route-modes', 'hard,topk,nomask',
    '--eval-seed', '42',
    '--output', str(EVAL_OUTPUT),
]
print('Running:', ' '.join(command))
subprocess.run(command, check=True)
print(f'✓ Evaluation saved to Kaggle Output: {EVAL_OUTPUT}')


In [ ]:
# Cell 6 — show saved result
import json
with open(EVAL_OUTPUT, 'r', encoding='utf-8') as f:
    evaluation = json.load(f)
print(json.dumps(evaluation, indent=2))
